# Operate - Alert Triage Agent

In this exercise, we build an agent that inspects a batch of monitoring alerts and decides which ones should be forwarded to the on-call engineer. Alerts that require human attention are sent to a Discord channel via an incoming webhook; informational or auto-resolving alerts are suppressed.

## Environment Variables & Imports

Before using this notebook, please ensure you have obtained an API Key from your LLM backend and update the `.env` file to include it as follows:

```bash
GOOGLE_API_KEY=<copy API key here>
OPENAI_API_KEY=<copy API key here>
ANTRHOPIC_API_KEY=<copy API key here>
LLAMA_CPP_API_KEY=<copy API key here>
```


Additionally, this notebook expects a `DISCORD_WEBHOOK_URL` entry in `.env` pointing at the incoming webhook of the on-call Discord channel.

In [ ]:
import initialize_notebook # noqa

In [ ]:
import json
import os
import pathlib

import jinja2

from hslu.dlm03.common import backend as backend_lib
from hslu.dlm03.common import chat as chat_lib
from hslu.dlm03.common import presets
from hslu.dlm03.common.displays import ipython_display
from hslu.dlm03.common import tools as tools_lib
from hslu.dlm03.util import file as file_lib

## Parameters

In [ ]:
backend_name = "Gemma 4 26B"
backend_config = presets.CONFIGS_BY_NAME[backend_name]
backend = backend_lib.LLM.from_config(backend_config)

In [ ]:
discord_webhook_url = os.environ["DISCORD_WEBHOOK_URL"]

# Current on-call engineer (used in the forwarded message).
oncall_engineer = {"name": "Vincent", "discord_handle": "@Vincent"}

# Sample batch of alerts coming from Alertmanager / Datadog / ...
alerts = [
    {
        "id": "A-1001",
        "severity": "info",
        "service": "api-gateway",
        "summary": "Deployment api-gateway@v2.3.1 completed successfully.",
        "fired_at": "2026-04-23T08:00:00Z",
    },
    {
        "id": "A-1002",
        "severity": "warning",
        "service": "checkout",
        "summary": "p95 latency above 500ms for 5m (current: 612ms).",
        "fired_at": "2026-04-23T08:03:20Z",
    },
    {
        "id": "A-1003",
        "severity": "critical",
        "service": "payments",
        "summary": "Error rate 42% on POST /charge for 2m - circuit breaker open.",
        "fired_at": "2026-04-23T08:07:11Z",
    },
    {
        "id": "A-1004",
        "severity": "info",
        "service": "batch-jobs",
        "summary": "Nightly ETL finished in 12m 30s (baseline: 11m).",
        "fired_at": "2026-04-23T08:10:00Z",
    },
    {
        "id": "A-1005",
        "severity": "critical",
        "service": "auth",
        "summary": "Auth service is returning 5xx on >10% of requests (login flow broken).",
        "fired_at": "2026-04-23T08:11:45Z",
    },
]

# MCP Server

The server exposes a small set of tools: list/get alerts, get on-call, forward to Discord, and mark-as-handled.

In [ ]:
import urllib.request
import urllib.error

from mcp.server import FastMCP

SERVER = FastMCP()

ALERTS_BY_ID = {alert["id"]: alert for alert in alerts}
FORWARDED: list[str] = []


@SERVER.tool()
def list_alerts() -> list[dict]:
    """Return the current batch of alerts (id, severity, service, summary, fired_at)."""
    return alerts


@SERVER.tool()
def get_alert(alert_id: str) -> dict | str:
    """Return the full detail for a single alert."""
    return ALERTS_BY_ID.get(alert_id, f"Unknown alert_id: {alert_id}")


@SERVER.tool()
def get_oncall() -> dict:
    """Return the current on-call engineer."""
    return oncall_engineer


import discord

@SERVER.tool()
def send_discord_message(content: str) -> str:
    """Send a message to the configured Discord channel via the incoming webhook.

    `content` supports Discord markdown. Keep it short and actionable (< 1800 chars).
    """
    if len(content) > 1800:
        return "Message too long - keep it under ~1800 characters."
    try:
        webhook = discord.SyncWebhook.from_url(discord_webhook_url)
        message = webhook.send(content=content, wait=True)
    except discord.HTTPException as e:
        return f"Discord webhook failed ({e.status}): {e.text}"
    except Exception as e:  # noqa: BLE001
        return f"Failed to send Discord message: {e}"
    FORWARDED.append(content)
    return f"Sent Discord message (id={message.id})."


@SERVER.tool()
def mark_alert_handled(alert_id: str, reason: str) -> str:
    """Record that an alert was intentionally NOT forwarded (auto-resolved, low-signal, ...)."""
    if alert_id not in ALERTS_BY_ID:
        return f"Unknown alert_id: {alert_id}"
    return f"Alert {alert_id} marked as handled. Reason: {reason}"

In [ ]:
import threading

import uvicorn

PORT = 5000
HOST = "localhost"

RUN_ARGS = {
    "app": SERVER.streamable_http_app,
    "port": PORT,
    "host": HOST,
    "log_level": "warning",
}

MCP_THREAD = threading.Thread(target=uvicorn.run, kwargs=RUN_ARGS, daemon=True)
MCP_THREAD.start()

MCP_URL = f"http://{HOST}:{PORT}/mcp"

# Agentic Harness

In [ ]:
import re
def remove_thoughts(text: str) -> str:
    # Remove well-formed <thought>...</thought> blocks (multiline safe)
    cleaned = re.sub(r"<thought>.*?</thought>", "", text, flags=re.DOTALL | re.IGNORECASE)

    # Optionally handle unclosed <thought> tags (strip until end)
    cleaned = re.sub(r"<thought>.*$", "", cleaned, flags=re.DOTALL | re.IGNORECASE)

    return cleaned.strip()

In [ ]:
system_instructions_template_string = \
"""You are the first-line on-call assistant. Your job is to triage incoming monitoring
alerts and decide, for each one, whether it needs to wake up the on-call engineer.

For every alert:
1. Inspect the alert (`get_alert` when you need more detail).
2. Decide one of two outcomes:
   - **Forward**: genuinely actionable, user-impacting, or sev-critical. Forward to the
     on-call engineer via `send_discord_message`. The message MUST include:
       * the severity and affected service,
       * a one-line summary of the incident,
       * the on-call engineer's Discord handle so they get pinged,
       * a short recommendation (e.g. what to check first).
   - **Suppress**: informational, expected noise, or auto-resolving. Call `mark_alert_handled`
     with a concise reason. Do NOT spam the on-call channel.
3. Be conservative: when in doubt, forward. Under-reacting to a real incident is worse than
   an extra ping.

Group related critical alerts into a single message if they clearly share a root cause.
"""
system_instructions_template = jinja2.Template(system_instructions_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message_template_string = \
"""Please triage the current batch of alerts and forward only the ones that need the
on-call engineer's attention."""
user_message_template = jinja2.Template(user_message_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
system_instructions = system_instructions_template.render()
user_message = user_message_template.render()

chat_display = ipython_display.IPythonChatDisplay()
chat_display.show()
chat = chat_lib.Chat(
    messages=[
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_message},
    ],
    observers=[chat_display],
)
async with tools_lib.mcp_session(MCP_URL) as session:
    await session.initialize()
    mcp_tools = await session.list_tools()
    tools = [tools_lib.tool_from_mcp(tool) for tool in mcp_tools.tools]
    done = False
    while not done:
        response = backend.generate(chat, tools=tools)
        done = True
        message = response.choices[0].message
        if message.content is not None:
            message.content = remove_thoughts(message.content)
        chat.append(message)
        for tool_call in message.tool_calls or ():
            done = False
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            tool_call_result = await session.call_tool(tool_name, arguments)
            for content in tool_call_result.content:
                tool_call_result_content = tools_lib.tool_call_result_from_mcp(
                    tool_call.id,
                    content,
                )
                chat.append(tool_call_result_content)